# **TP LLM : Large Language Models**


## **Introduction**


IA générative, LLM, ChatGPT, agents... le vocabulaire associé à ces technologies est dense et souvent flou. Voici un tableau qui résume et distingue les principaux composants que l'on peut retrouver au sein d'une application d'IA générative :

**Composants d'une application d'IA générative :**

| Composant                  | Analogie véhicule  | Rôle                                                                                   | Exemples                                                      |
| -------------------------- | ------------------ | -------------------------------------------------------------------------------------- | ------------------------------------------------------------- |
| LLM                        | Moteur             | Coeur du système : modèle statistique qui comprend et génère du texte.                 | GPT-5.4 / Mistral Large 3 / Claude Sonnet 4.6 / DeepSeek V3.2 |
| Interface utilisateur (UI) | Tableau de bord    | Ce que l'utilisateur voit et utilise pour interagir avec le LLM.                       | ChatGPT / Le Chat / WattElse                                  |
| Outils                     | Options            | Permettent d'enrichir les capacités de base.                                           | Recherche web / Exécution de code / Génération d'images / RAG |
| Agent                      | Pilote automatique | Coordonne le LLM et les outils pour réaliser des tâches complexes en plusieurs étapes. | LangGraph / CrewAI / Microsoft AutoGen                        |

L'objectif de ce TP est d'explorer et de comprendre le fonctionnement des **LLMs**, moteurs des applications d'IA générative, afin de mieux appréhender leurs enjeux et leurs limites.

**Plan du TP :**

1. Choix et chargement du LLM
2. Exploration du fonctionnement du LLM
3. Comment transformer un LLM en assistant ?
4. (Bonus) Tool Calls : vers l'IA agentique


## **0. Installations, imports et définition de fonctions**


### Imports et paramètres


In [ ]:
import json
import os
import urllib.parse
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import torch
from sklearn.decomposition import PCA
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TextStreamer,
    logging,
    pipeline,
)

# Set device and corresponding data type
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

# Disable torch gradient tracking
torch.set_grad_enabled(False)

# Hide warnings
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
logging.set_verbosity_error()

### **Clé d'API**


In [ ]:
os.environ["HF_TOKEN"] = ""  # ask for your key!

### Définition des fonctions


In [ ]:
def plot_token_embeddings(token_ids: list[int]) -> None:
    """
    Plot 2D representations of token embeddings with vectors using Plotly.
    """
    # 1. Extract embeddings
    with torch.no_grad():
        # Get embeddings and move to CPU immediately for processing
        embeddings = model.get_input_embeddings()(
            torch.tensor(token_ids).to(model.device)
        )
        embeddings_np = embeddings.cpu().float().numpy()

    # 2. Reduce dimensions to 2D
    pca = PCA(n_components=2)
    coords = pca.fit_transform(embeddings_np)

    # 3. Prepare data for Plotly
    df = pd.DataFrame(
        {
            "x": coords[:, 0],
            "y": coords[:, 1],
            "token": [tokenizer.decode([t]) for t in token_ids],
        }
    )

    # 4. Create the scatter plot
    fig = px.scatter(
        df,
        x="x",
        y="y",
        text="token",
        title="2D Token Embeddings (PCA)",
        template="plotly_white",
    )

    # 5. Add vectors (arrows) and origin lines
    for i in range(len(df)):
        fig.add_annotation(
            ax=0,
            ay=0,  # Arrow starts at origin
            axref="x",
            ayref="y",  # Use coordinate system for start
            x=df.x[i],
            y=df.y[i],  # Arrow ends at point
            xref="x",
            yref="y",  # Use coordinate system for end
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor="blue",
            opacity=0.6,
        )

    # Clean up layout: add origin lines and adjust text position
    fig.update_traces(textposition="top center")
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)

    fig.show()


def measure_memory(token_count: int) -> float:
    """
    Returns total GPU memory allocated for processing `token_count` tokens (in MB).
    """
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    dummy_token_id = tokenizer.encode("the")[0]
    input_ids = torch.tensor([[dummy_token_id] * token_count], device="cuda")
    with torch.no_grad():
        _ = model(input_ids)
    return torch.cuda.max_memory_allocated() / 1024**2  # MB


WMO_CODES = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    56: "Light freezing drizzle",
    57: "Heavy freezing drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    66: "Light freezing rain",
    67: "Heavy freezing rain",
    71: "Slight snowfall",
    73: "Moderate snowfall",
    75: "Heavy snowfall",
    77: "Snow grains",
    80: "Slight rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    85: "Slight snow showers",
    86: "Heavy snow showers",
    95: "Thunderstorm",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail",
}


## **1. Choix et chargement du LLM**


En pratique, qu'est-ce qu'un LLM ?

Un LLM est un **réseau de neurones artificiels**, comme vous avez pu les découvrir plus tôt dans cette formation ou dans [DATAS2](https://propulse.rte-france.com/Catalog/TrainingShops/TrainingView.aspx?idTraining=140902456), avec une architecture particulière nommée [Transformer](https://fr.wikipedia.org/wiki/Transformeur) qui ne sera pas abordée dans ce TP.

Concrètement, un LLM n'est rien de plus qu'un un gros fichier qui contient l'ensemble des **poids** (aussi appelés **paramètres**) du réseau de neurones. Le nombre de paramètres d'un LLM et la taille du fichier qui le représente varient grandement en fonction des modèles.

Voici, pour quelques LLMs, le nombre de paramètres ainsi que la taille du fichier associé :

| **LLM**     | **Nombre de paramètres** | **Taille du fichier** | **Commentaire**                                                                                                                                                     |
| ----------- | ------------------------ | --------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Mistral-7B  | 7 milliards              | 14 Go                 | Modèle à l'origine du succès de Mistral AI.                                                                                                                         |
| gemma-4-31B | 33 milliards             | 66 Go                 | Exemple de la gamme _open-weight_ des modèles développés par Google. Excellent rapport taille/performance.                                                          |
| GPT-3       | 175 milliards            | 350 Go                | Modèle dernière ChatGPT lors de son lancement en novembre 2022.                                                                                                     |
| DeepSeek-R1 | 685 milliards            | 1370 Go               | Modèle chinois qui a beaucoup fait parlé de lui en janvier 2025 : performances comparables aux meilleurs modèles américains, à une fraction du coût d'entraînement. |

Certaines entreprises publient leurs LLMs en _open-weight_ (Meta, Mistral, DeepSeek...), ce qui permet de les télécharger et de les exécuter sur un PC pour les plus petits ou sur un serveur de calcul pour les plus gros. D'autres (comme OpenAI ou Anthropic) proposent uniquement un accès par API à leurs modèles, ce qui rend toute utilisation locale (sur PC ou serveur propre) impossible.

> On distingue **_open-weight_** et **_open-source_** pour les LLMs.
> Un modèle _open-weight_ rend simplement ses poids publics (le modèle entraîné, utilisable tel quel).
> Un vrai modèle _open-source_ va plus loin : il publie aussi les données et le code d'entraînement,
> de façon à ce que n'importe qui puisse le recréer de zéro — ce que très peu d'entreprises font.


### **1.1 Découverte d'Hugging Face 🤗**


[Hugging Face 🤗](https://huggingface.co/) est une plate-forme permettant le partage de modèles, de code et de données dans le domaine du **NLP (Natural Language Processing)**, et plus marginalement dans d'autres domaines de l'intelligence artificielle (vision par ordinateur, apprentissage par renforcement). Le site permet à n'importe qui, du simple utilisateur aux grandes entreprises comme **Meta** ou **Mistral**, de partager son travail pour qu'il soit utilisable par la communauté.

> RTE possède un compte Hugging Face ! Vous pouvez y accéder [ici](https://huggingface.co/rte-france).

Pour explorer les modèles disponibles, rendez-vous dans l'onglet [models](https://huggingface.co/models).

Pour ce TP, nous allons utiliser un "petit" LLM : [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B), développé par l'entreprise Chinoise **Alibaba**. Il est composé de **2 milliards** de paramètres et pèse **4 Go**.


### **1.2 Chargement du modèle et de son tokenizer**


Nous utilisons la librairie python [transformers](https://github.com/huggingface/transformers), développée par Hugging Face, qui permet de charger et d'utiliser facilement les modèles disponibles sur la plateforme.


In [ ]:
model_name = "Qwen/Qwen3-1.7B"

In [ ]:
# Download tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Download model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=DTYPE,
    device_map=DEVICE,
    trust_remote_code=True,
)

In [ ]:
# Put model into inference mode
model.eval()

## **2. Exploration du fonctionnement du LLM**


Fondamentalement, un LLM ne sait faire qu'une seule chose : générer du texte mot par mot, ou plutôt token par token (nous verrons la distinction par la suite).

**Un LLM peut être vu comme un gros modèle d'auto-complétion : à partir d'un texte existant (le prompt), il prédit le mot suivant en fonction du contexte.**

Dans cette section, nous allons explorer comment un LLM fonctionne pour prédire le token suivant. Dans la section suivante, nous expliquerons comment, à partir de ce fonctionnement simpliste, il est possible de simuler un assistant qui répond à nos questions.

Le processus de génération du token suivant peut être divisé en 3 étapes :

- **tokenizer** : découpe le prompt en une séquence de tokens
- **embedding** : transforme chaque token en un vecteur numérique que le LLM peut interpréter
- **réseau de neurones** : prédit le token suivant


### **2.1 Tokenizer**


Le tokenizer est un algorithme séparé du LLM qui peut être vu comme une étape de pré-traitement du prompt. Chaque LLM possède son propre tokenizer. Son objectif est de décomposer le prompt en une série de tokens.

Le tokenizer possède un nombre fini de tokens que l'on appelle le **vocabulaire** du tokenizer (et du LLM par extension).

Avant d'être transmis au LLM, le prompt est décomposé par le tokenizer en une séquence de tokens sélectionnés parmi ceux disponibles dans son vocabulaire.


In [ ]:
# Show LLM vocabulary size
print(f"Vocabulary size: {tokenizer.vocab_size}")

In [ ]:
# Define a prompt
prompt = "RTE est le gestionnaire du réseau de transport d'électricité."

# Try tokenizer functions
tokenized = tokenizer.tokenize(prompt)
encoded = tokenizer.encode(prompt, add_special_tokens=False)

# Check differences between each function
print(f"Tokenized : {[tokenizer.decode(token) for token in encoded]}")
print(f"Encoded : {encoded}")

---

**Question**

Rendez-vous sur [tiktokenizer/Qwen2.5-72B](https://tiktokenizer.vercel.app/?model=Qwen%2FQwen2.5-72B) pour essayer le tokenizer de manière plus interactive.

Essayez également le tokenizer d'un LLM à l'état de l'art, par exemple celui des modèles `GPT-5.X` d'OpenAI :  [GPT-5.X tokenizer](https://platform.openai.com/tokenizer).

- Que remarquez-vous ?
- Quelles conséquences sur le traitement et la génération de tokens par le LLM ?

---


### **2.2 LLM : Embedding**


Un réseau de neurones accepte uniquement des données numériques en entrée. Chaque token est donc converti en un vecteur numérique, aussi appelé **_embedding_**. Son rôle est de représenter la sémantique associée au token.

Les _embeddings_ possèdent la propriété suivante : deux tokens qui ont un sens proche sont représentés par des vecteurs numériques similaires.

La première couche du LLM est une simple table de correspondance qui associe à chaque token un embedding.


In [ ]:
# Print LLM embedding layer shape
model.get_input_embeddings()

In [ ]:
# Define a list of words
word_list = "cat dog green blue house room"

# Plot tokens embeddings in a reduce 2D space
prompt_tokens_ids = list(set(tokenizer.encode(word_list, add_special_tokens=False)))
plot_token_embeddings(prompt_tokens_ids)

---

**Question**

Ajoutez des mots à la liste et observez le résultat.

- Quelles limites anticipez-vous avec cette manière de représenter les mots ?
- À quoi va servir le réseau de neurones (de plusieurs milliards de paramètres !) qui vient ensuite ?

---


### **2.3 LLM : Réseau de neurones**


Le réseau de neurones possède une architecture particulière : l'architecture **Transformer**.

Le rôle du Transformer est de convertir les embeddings statiques en embeddings contextuels. Il analyse les dépendances entre tokens et recalcule la valeur de chaque vecteur pour qu'elle représente la sémantique globale de la séquence.

Cette transformation des embeddings permet de générer des tokens de manière plus contextualisée et est à l'origine des comportements émergents qu'on observe avec les LLMs : des capacités de raisonnement ou de résolution de problèmes complexes qui n'étaient **pas explicitement programmées**, mais qui découlent de cette compréhension profonde des structures du langage.

D'un point de vue fonctionnel, ce réseau de neurones peut-être décrit par le couple entrée/sortie suivant :

- Entrée : liste de vecteurs numériques, chacun représentant un token de notre prompt
- Sortie : densité de probabilité sur l'ensemble des tokens du vocabulaire, qui associe à chaque token la probabilité d'être le suivant


In [ ]:
# Define prompt, temperature, and top_k
prompt = "The capital of Australia is"
temperature = 0.5
top_k = 10

# Tokenize prompt
inputs = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

# Make a forward pass
output = model.forward(input_ids=inputs, use_cache=False)["logits"].squeeze()[-1]

# Transform model output to probability distribution using softmax
output_softmax = torch.nn.functional.softmax(output / temperature, dim=0)

# Filter top_k tokens
top_k_indices = torch.topk(output_softmax, top_k).indices
top_k_probabilities = output_softmax[top_k_indices].cpu().detach().float().numpy()
top_k_tokens = [tokenizer.decode([token_id]) for token_id in top_k_indices]

# Barplot
px.bar(
    x=top_k_tokens,
    y=top_k_probabilities,
    labels={"x": "Tokens", "y": "Probability"},
    title=prompt + "...",
)

---

**Question**

Modifiez le paramètre `temperature` entre 0 et 1.

- Quelle est son influence sur la distribution de sortie ?

Modifiez le paramètre `top_k`.

- Quel est son rôle ? Pourquoi en a-t-on besoin ?

---


### **2.4 Pipeline de génération**


C'est (presque) tout ce qu'il y a à savoir sur le fonctionnement d'un LLM !

Pour générer du texte, il suffit de découper notre prompt initial en une liste de tokens grâce au tokenizer puis de répéter le processus suivant :

- lancer une inférence pour générer la densité de probabilité
- échantillonner cette densité pour sélectionner un token
- ajouter ce token à la séquence de tokens d'entrée

La librairie `transformers` propose des fonctions pour réaliser ce processus facilement et générer du texte :


In [ ]:
# Generation pipeline to chain model predictions
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    streamer=TextStreamer(
        tokenizer, skip_special_tokens=True
    ),  # to print tokens on the fly
)

In [ ]:
# Define prompt
prompt = "The capital of Australia is"

# Generate text continuation
prediction = generator(
    prompt,
    do_sample=True,
    temperature=0.5,
    max_new_tokens=30,
)

---

**Question**

Relancez plusieurs fois la génération en modifiant le paramètre `temperature` et observez le texte généré.

- Dans quels cas utiliser une température élevée ?
- Dans quels cas utiliser une température faible ?

---


### **2.5 Un point sur la mémoire**


Les LLMs, de par leur nombre de paramètres important, sont très gourmants en **mémoire**.

Pour générer du texte avec un LLM, il faut le charger **entièrement** en mémoire (rappelez-vous de la taille des fichiers). Mais ce n'est pas tout ! Pendant l'inférence, le LLM garde également les embeddings et d'autres valeurs numériques associées à chaque token en mémoire.

La taille du prompt impact donc directement la quantité de mémoire nécessaire pour générer du texte. La quantité de mémoire totale est une fonction du type :

$$Mémoire = Taille\ LLM + f(longueur\ du\ prompt)$$

_La cellule suivante ne fonctionne que si le LLM est chargé sur GPU._


In [ ]:
if torch.cuda.is_available():
    # Compute memory
    token_counts = [2**i for i in range(0, 13)]
    memories = [measure_memory(n) for n in token_counts]

    px.line(
        x=token_counts,
        y=memories,
        markers=True,
        labels={"x": "Number of prompt tokens", "y": "Total Memory Used (MB)"},
        title="Total Memory Used vs Prompt Token Count",
    ).show()
else:
    print("No GPU available.")

---

**Question**

Observez la tarification appliquée par OpenAI lorque l'on utilise leurs LLMs via API : [OpenAI API Pricing](https://developers.openai.com/api/docs/pricing)

- Pourquoi est-on facturé au token et pas à la question ?
- Pourquoi y a-t-il une tarification différenciée entre *"short context"* et *"long context"* ?
- Question annexe : pourquoi y a-t-il une différence de prix entre *"input tokens"* et *"output tokens"* ?

---


## **3. Comment transformer un LLM en assistant ?**


Un LLM n'est qu'un moteur à générer du texte, un gros modèle d'auto-complétion. Ce n'est pas encore un assistant qui répond à nos questions ! Dans l'exemple suivant, observez comme le LLM complète notre prompt par un texte probable (un message sur un blog) sans répondre à la question.


In [ ]:
# Define prompt
prompt = "How to install numpy ?"

# Generate text continuation
prediction = generator(prompt, do_sample=False, max_new_tokens=30)

Pour structurer l'interaction avec un LLM et l'inciter à répondre de manière spécifique, on utilise une méthode nommée **Chat Prompt Template**. C'est une technique fondamentale intégrée à la plupart des interfaces de chatbots (comme ChatGPT ou Claude) qui organise le dialogue en différents rôles. Le principe consiste à structurer le prompt en messages distincts, chacun associé à un rôle précis :

- **System** : définit le comportement global du modèle, ses instructions de fond, sa personnalité ou ses contraintes. Ce message est généralement invisible pour l'utilisateur final mais conditionne toute la conversation.
- **User** : représente les messages envoyés par l'utilisateur.
- **Assistant** : représente les réponses générées par le modèle.

Le Chat Prompt Template transforme une simple question en une structure qui guide le modèle et facilite l'interaction avec le LLM. Cela permet de mieux contrôler le texte généré, mais aussi d'injecter du contexte métier, des règles de formatage, un ton particulier, etc.

Une structure typique est la suivante :

```text
<system>
Below is a conversation between a user and you (assistant).
You give helpful and concise answers to the user questions.
</system>

<user>
Question 1.
</user>

<assistant>
Answer 1.
</assistant>

<user>
Question 2.
</user>

<assistant>
Answer 2.
</assistant>

...

```

Pour simuler un assistant IA :

- Un utilisateur entre une question
- Un Chat Prompt Template est créé en insérant la dernière question de l'utilisateur, jusqu'à la balise **\<assistant>**
- Le LLM génère la suite jusqu'à ce que le tokens **\</assistant>** soit échantillonné

Ce processus est répété à chaque question de l'utilisateur.


In [ ]:
prompt = """
<system>
Below is a conversation between a user and you (assistant).
You give helpful and concise answers to the user questions.
</system>

<user>
How to install numpy?
</user>

<assistant>
"""  # LLM starts generating tokens from this <assistant> tag

# Generate text continuation
output = generator(
    prompt,
    do_sample=True,
    temperature=0.1,
    tokenizer=tokenizer,
    max_new_tokens=100,
    stop_strings="</assistant>",
)

Quand vous envoyez un message dans ChatGPT, Claude, WattElse ou autre, votre message est inséré dans un Chat Prompt Template.

Ce qui est affiché dans l'interface utilisateur (vos questions et les réponses de l'IA) est simplement une version plus lisible de ce dernier, avec le prompt système masqué.

> Les entreprises masquent volontairement le prompt système, notamment pour les raisons suivantes :
>
> - Sécurité : si un utilisateur malveillant connaît exactement les instructions système, il peut plus facilement chercher à les contourner (_prompt injection_, _jailbreak_) pour générer du contenu dangereux ou illégal. Le garder secret constitue une première ligne de défense.
> - Propriété intellectuelle : le prompt système représente souvent un travail d'ingénierie conséquent. Les entreprises y investissent du temps pour affiner le comportement du modèle, définir sa personnalité, ses limites, son ton. Le divulguer reviendrait à exposer un avantage concurrentiel.


---

**Question**

Modifiez le prompt système et la temperature et analysez le comportement du LLM. Essayez par exemple de le faire répondre dans un format structuré comme JSON.

- Quelles techniques supplémentaires pouvez-vous imaginer pour que le LLM respecte mieux vos instructions ?

Supprimez le paramètre `stop_strings` lors de la génération.

- À quoi sert ce paramètre ? Pourquoi en a-t-on besoin ?

Les LLMs n'ont aucune mémoire : pour gérer l'historique d'une conversation, il est nécessaire de renvoyer, à chaque question de l'utilisateur, l'ensemble du prompt système et des questions/réponses précédentes.

- Quels impacts cela peut-il avoir :
  - Sur la qualité des réponses ?
  - Sur le coût en calcul ?

---


## **4. (Bonus) Tool Calls : vers l'IA agentique**


Comme nous l'avons vu, un LLM est un modèle statistique figé dans le temps. Par défaut, il ne peut pas accéder à Internet, ne connaît pas l'heure actuelle, ne peut pas exécuter du code, ni interagir avec des systèmes externes. Pour combler ces lacunes, on peut lui donner accès à des **outils**.

Le principe du _Tool Calling_ (ou _Function Calling_) est le suivant : on explique au LLM, via son prompt système, qu'il a des fonctions à sa disposition, ce qu'elles font, et comment il doit structurer sa réponse s'il veut s'en servir (souvent en générant un format spécifique comme du JSON).

**Attention :** Le LLM n'exécute pas le code lui-même ! Il se contente de générer le texte demandant l'exécution de l'outil. C'est l'application autour du LLM (votre code Python) qui intercepte cette demande, exécute la fonction, et renvoie le résultat au LLM pour qu'il puisse formuler sa réponse finale.


### **4.1 Un premier outil**


Commençons par définir un outil via une fonction python.

La fonction suivante permet de récupérer des prévisions météo du lendemain grâce à l'API [Open Meteo](https://open-meteo.com/).


In [ ]:
def get_tomorrow_weather(city: str) -> dict:
    """
    Return tomorrow's meteo forecast for a specific city.
    """
    # 1. Geocoding: City name → coordinates
    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={urllib.parse.quote(city)}&count=1"
    with urllib.request.urlopen(geo_url) as r:
        geo = json.loads(r.read())

    if not geo.get("results"):
        return {"error": f"Ville '{city}' introuvable"}

    loc = geo["results"][0]
    lat, lon = loc["latitude"], loc["longitude"]

    # 2. Fetch Forecast
    weather_url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        f"&daily=temperature_2m_max,temperature_2m_min,weathercode"
        f"&timezone=auto"
    )

    with urllib.request.urlopen(weather_url) as r:
        weather = json.loads(r.read())

    # 3. Extract Tomorrow's Data (Index 1)
    daily = weather["daily"]
    tomorrow_code = daily["weathercode"][1]

    return {
        "city": loc["name"],
        "date": daily["time"][1],
        "temp_max": f"{daily['temperature_2m_max'][1]}°C",
        "temp_min": f"{daily['temperature_2m_min'][1]}°C",
        "condition": WMO_CODES.get(tomorrow_code, f"Code {tomorrow_code}"),
    }

In [ ]:
# Test tool
get_tomorrow_weather("Paris")

Voyons maintenant comment forcer le LLM à utiliser cet outil en lui donnant des instructions claires dans son **prompt système**.


In [ ]:
prompt = """
<system>
Below is a conversation between a user and you (assistant).
You give helpful and concise answers to the user questions.

You have access to the following tool:
- get_tomorrow_weather(city: str): Return tomorrow's meteo forecast for a specific city.

You must respond strictly in this exact JSON format to use the tool:
{"tool": "get_tomorrow_weather", "city": "City Name"}
</system>

<user>
What's the weather like in Paris tomorrow?
</user>

<assistant>
"""

# Generate text continuation
output = generator(
    prompt,
    do_sample=True,
    temperature=0.1,
    tokenizer=tokenizer,
    max_new_tokens=100,
    stop_strings="</assistant>",
)

---

**Question**

Relancez la cellule précédente en posant une question sur un sujet autre que la météo.

- Que se passe-t-il ?

Mettez à jour le prompt système pour corriger ce comportement et testez à nouveau votre question.

---


### **4.2 ReAct : la boucle complète**

Pour automatiser ce processus, nous avons besoin d'une boucle logique (le cœur de ce qu'on appelle un Agent IA) :

1. (**Analyse**) Le LLM reçoit la question de l'utilisateur.
2. (**Cycle décision/action**) Le LLM génère une réponse et décide soit :
   - D'appeler un outil (JSON) : notre code Python intercepte ce JSON, exécute la fonction associée, et renvoie le résultat au LLM qui pourra prendre une nouvelle décision. On retourne au début de l'étape 2.
   - De générer une réponse finale (texte simple), la boucle s'arrête et la réponse est renvoyée à l'utilisateur.

Ce mécanisme permet au LLM d'enchaîner plusieurs appels d'outils avant de formuler sa réponse finale

Le processus décrit ci-dessus est une version simplifiée d'une approche plus connue sous le nom de [ReAct](https://www.promptingguide.ai/techniques/react) (pour Reasoning + Acting), à la base des systèmes agentiques les plus complexes.

Une implémentation simple est proposée ci-dessous, via la fonction `run_agent`.

> En pratique, il existe des librairies qui permettent d'implémenter des agents de manière bien plus efficace, par exemple [LangGraph](https://www.langchain.com/langgraph) ou [OpenAI Agents SDK](https://developers.openai.com/api/docs/guides/agents).


In [ ]:
# Dict holding tools definition
TOOLS_REGISTRY = {
    "get_tomorrow_weather": {
        "func": get_tomorrow_weather,
        "description": "Returns tomorrow's meteo forecast for a specific city",
        "args": [{"name": "city", "type": "str"}],
    }
}

In [ ]:
# Textual description of tools list
tools_list = "\n".join(
    [
        f"- {name}({', '.join(f'{arg['name']}: {arg['type']}' for arg in info['args'])}): {info['description']}"
        for name, info in TOOLS_REGISTRY.items()
    ]
)

# Define a system prompt
system_prompt = f"""<system>
Below is a conversation between a user and you (assistant).
You give helpful and concise answers to the user questions.
You have access to the following tools:

{tools_list}

To use a tool, you MUST reply with ONLY this JSON: {{"tool": "name", "args": {{"arg_name": "value"}}}}
If no tool is needed, answer normally.
</system>\n"""

print(system_prompt)

In [ ]:
def run_agent(question: str, max_iterations: int = 5):
    """
    Simple implementation of a ReAct loop, with all intermediate interations printed.
    """
    # Build initial chat prompt template
    prompt = f"{system_prompt}<user>\n{question}\n</user>\n<assistant>\n"

    # Start ReAct loop
    for i in range(max_iterations):
        print(f"\n{'=' * 55}")
        print(f"--- Iteration {i + 1} ---")

        # 1. Generate LLM response
        raw_res = (
            generator(
                prompt,
                do_sample=False,
                tokenizer=tokenizer,
                max_new_tokens=150,
                stop_strings=["</assistant>"],
                return_full_text=False,
            )[0]["generated_text"]
            .strip()
            .split("</assistant>")[0]
        )

        # 2. Check if the LLM wants to call a tool
        if '{"tool":' in raw_res:
            # Parse and execute tool call
            try:
                call = json.loads(raw_res)
            except json.JSONDecodeError:
                print("❌ Error: AI generated invalid JSON.")
                return None

            tool_name = call["tool"]
            arguments = call.get("args", {})

            if tool_name not in TOOLS_REGISTRY:
                print(f"❌ Error: Tool '{tool_name}' not found.")
                return None

            print(f"🛠️  Calling '{tool_name}' with {arguments}")
            result = TOOLS_REGISTRY[tool_name]["func"](**arguments)
            print(f"📤 Result: {result}")

            # Append tool call + result to prompt and continue the loop
            prompt += f"{raw_res}</assistant>\n<tool_result>\n{result}\n</tool_result>\n<assistant>\n"
        # 3. If no tool call found, exit as it is the final LLM answer
        else:
            return

    print("⚠️ Max iterations reached.")
    return None

Utilisez la cellule suivante pour tester l'agent sur plusieurs questions et analysez les différentes itérations. Demandez par exemple la météo de plusieurs villes.


In [ ]:
# Test the agent
run_agent("How should I dress tomorrow if I'm going for a walk in Lyon?")

---

**Question**

Créez une (ou plusieurs !) nouvelle(s) fonction(s) Python. Quelques exemples pour s'inspirer :
  - `get_random_number`: renvoie un nombre aléatoire.
  - `get_weekday`: renvoie le nom du jour de la semaine de n'importe quelle date.
  - `get_ISS_position`: renvoie la position exacte de la Station Spaciale Internationale en utilisant l'API [ISS-Location-Now](http://open-notify.org/Open-Notify-API/ISS-Location-Now/)

Ajoutez-les dans le dictionnaire `TOOLS_REGISTRY`.

Testez l'assistant sur des questions qui nécessitent d'utiliser ces nouvelles fonctions :
  - Des questions simples qui nécessistent l'utilisation d'un seul outil (le LLM doit simplement choisir le bon outil)
  - Des questions complexes qui nécessistent l'appel à plusieurs outils avant de répondre

---


---

**Question bonus**

Créez une fonction qui prend en entrée un script python sous forme de `str`, l'execute via la commande [`exec`](https://docs.python.org/3/library/functions.html#exec) et renvoie la sortie standard (`stdout`, les `print()` dans le code).

Ajoutez cette fonction dans le dictionnaire `TOOLS_REGISTRY` et testez l'agent sur de nouvelles questions.

- Quels risques anticipez-vous avec ce type d'outil ?
- Quelles solutions peut-on imaginer ?

---
